# Filamentation 1030 nm / 13 µJ -- visualisation (fluence, I_max, rho_e, rho_s, cycles, L_c)

Ce notebook utilise le solveur `NewSim3juillet.py` (copié depuis les fichiers
uploadés, dossier `sim_1030nm_experiment/`), PAS le paquet modulaire
`sim/filament_sim.py` utilisé par `term_ablation_study.ipynb` -- ce sont deux
implémentations distinctes du même modèle physique, avec des interrupteurs
différents (`NewSim3juillet.py` n'a que 3 interrupteurs grossiers
`enable_kerr`/`enable_avalanche`/`enable_recombination`, pas les 6 termes fins
de l'éq. (3) du rapport).

**Géométrie et énergie reprises telles quelles de `notebooksimu3juillet.ipynb`
(cellule 2) et cross-vérifiées contre `unified_filament_slider_v3.py`** : les
deux fichiers calculent indépendamment `Z_FOCUS_GLASS_DIST_UM = N_GLASS *
Z_FOCUS_AIR_DIST_UM = 1.45 * 272 = 394.4 µm`, et le premier l'arrondit à
`394.0` -- accord à 0.1%, donc pas une simple copie, un vrai recoupement.

**Un point a été corrigé par rapport à `notebooksimu3juillet.ipynb` : voir la
cellule `n2` ci-dessous.**

In [ ]:
import sys, json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.constants import c as c_SI, epsilon_0, m_e, elementary_charge as q_e

sys.path.insert(0, str(Path.cwd().parent / "sim_1030nm_experiment"))
sys.path.insert(0, str(Path.cwd() / "sim_1030nm_experiment"))

from NewSim3juillet import run, n_sellmeier, SELLMEIER_B, SELLMEIER_L2

OUT_ROOT = Path("runs_1030nm")
OUT_ROOT.mkdir(exist_ok=True)
FIG_DIR = OUT_ROOT / "figures_for_report"
FIG_DIR.mkdir(exist_ok=True)

print("Imports OK")

## 1. Paramètres -- géométrie, énergie, matériau

### Géométrie (interface -> foyer gaussien)
Reprise exacte de `notebooksimu3juillet.ipynb` cellule 2 : le foyer géométrique
est à `Z_FOCUS_AIR_DIST_UM=272 µm` *dans l'air* avant réfraction ; une fois
dans le verre (indice de groupe pompe `N_GLASS=1.45`), il recule à
`Z_FOCUS_GLASS_DIST_UM = N_GLASS * 272 = 394 µm` sous l'interface. Le solveur
travaille dans un repère centré sur ce foyer (`z_sim=0`), donc la face
d'entrée est à `begin = -394 µm`.

### Énergie -- correction Fresnel + enveloppe temporelle
`ENERGY_INPUT_UJ=13.0` est l'énergie *avant* l'interface air/verre. Deux
corrections sont nécessaires avant de la passer à `run(energy_uJ=...)` :
1. **Transmission de Fresnel** en incidence normale,
   `T = 1 - ((n0-1)/(n0+1))^2`, pour obtenir l'énergie réellement couplée
   dans le verre.
2. **Convention flat-top de `Config.__post_init__`** : la formule
   `I0 = 2*energy_uJ/(pi*w0^2*delta_t)` traite implicitement `delta_t` comme
   une durée flat-top, alors que le profil réel est gaussien temporel de FWHM
   `delta_t`. Pour qu'`energy_uJ` redonne la vraie énergie physique une fois
   intégrée sur le profil gaussien, il faut la diviser par
   `sqrt(pi/(4 ln 2)) ~ 1.0645` avant de l'injecter dans `run()`.

### n2 -- correction apportée ici
`notebooksimu3juillet.ipynb` ne surchargeait PAS `n2` : le run utilisait donc
la valeur par défaut de `Config`, `n2=2.4e-20 m^2/W`, qui est la valeur
mesurée à **800 nm** (Taylor et al., compilation Milam 1998 -- c'est
exactement la valeur déjà citée Section 8 du rapport pour 800 nm). Cette
expérience est à **1030 nm** : la valeur la plus proche disponible dans
Milam 1998 est `n2 ~ 2.74e-20 m^2/W` à 1053 nm, déjà utilisée Section 8 pour
calculer `P_cr ~ 4.0 MW` à 1030 nm. Ci-dessous les deux valeurs sont gardées
en parallèle (`n2_default` = ce que `notebooksimu3juillet.ipynb` utilisait
réellement, `n2_1030nm` = la valeur correcte pour cette longueur d'onde) pour
que l'écart soit visible sur les figures plutôt que silencieusement corrigé.

In [ ]:
# --- Géométrie (identique notebooksimu3juillet.ipynb cellule 2) ---
Z_FOCUS_AIR_DIST_UM   = 272.0
N_GLASS               = 1.45
Z_FOCUS_GLASS_DIST_UM = N_GLASS * Z_FOCUS_AIR_DIST_UM   # 394.4 um
BEGIN_M = -Z_FOCUS_GLASS_DIST_UM * 1e-6
END_M   =  800e-6

# --- Energie (identique notebooksimu3juillet.ipynb cellule 2) ---
N0_PUMP         = 1.4500
ENERGY_INPUT_UJ = 13.0
TRANSMISSION    = 1.0 - ((N0_PUMP - 1.0) / (N0_PUMP + 1.0))**2
ENERGY_IN_GLASS = ENERGY_INPUT_UJ * TRANSMISSION
GAUSS_FLATTOP   = float(np.sqrt(np.pi / (4.0 * np.log(2))))
ENERGY_SIM_UJ   = ENERGY_IN_GLASS / GAUSS_FLATTOP

# --- Laser (identique notebooksimu3juillet.ipynb cellule 2) ---
WAVELENGTH_M = 1030e-9
W0_M         = 3e-6
DELTA_T_S    = 263e-15

# --- Materiau : deux valeurs de n2 en parallele (voir markdown ci-dessus) ---
N2_DEFAULT_M2W = 2.4e-20    # Config default = valeur Milam 1998 a 800 nm (non corrigee)
N2_1030NM_M2W  = 2.74e-20   # Milam 1998 a 1053 nm, deja utilisee Section 8 du rapport
UI_EV          = 9.0        # Config default, coherent avec le reste du rapport
RHO_MAX_CM3    = 2.1e22

# --- Grille (identique notebooksimu3juillet.ipynb cellule 2) ---
LZ = END_M - BEGIN_M
NZ = int(LZ / 24e-9)
NT = 2000
NR = 3001
R_FACTOR = 90.0

n0_1030 = n_sellmeier(WAVELENGTH_M)

print(f"Focus verre        : {Z_FOCUS_GLASS_DIST_UM:.1f} um sous l'interface")
print(f"Boite sim           : z_sim in [{BEGIN_M*1e6:+.1f}, {END_M*1e6:+.1f}] um  (Nz={NZ}, dz={LZ/NZ*1e9:.1f} nm)")
print(f"Transmission Fresnel: {TRANSMISSION*100:.2f} %  ({ENERGY_INPUT_UJ:.2f} -> {ENERGY_IN_GLASS:.2f} uJ dans le verre)")
print(f"energy_uJ envoye a run(): {ENERGY_SIM_UJ:.3f} uJ  (facteur flat-top/gaussien = {GAUSS_FLATTOP:.4f})")
print(f"n0(1030nm) = {n0_1030:.4f}  (Sellmeier, coherent avec N0_PUMP={N0_PUMP})")
print(f"n2_default = {N2_DEFAULT_M2W:.2e} m^2/W (800 nm, PAS corrige pour 1030 nm)")
print(f"n2_1030nm  = {N2_1030NM_M2W:.2e} m^2/W (1053 nm, valeur la plus proche dispo)")

## 2. Puissance critique et longueur de collapse -- prédiction avant de lancer

Mêmes formules que Section 8 du rapport (Marburger/Dawes) :
$$P_{cr} = \frac{3.77\,\lambda_0^2}{8\pi n_0 n_2}, \qquad
L_c = \frac{0.367\,L_{DF}}{\sqrt{[(P_{in}/P_{cr})^{1/2}-0.852]^2-0.0219}}, \qquad
\frac{1}{L_{c,f}} = \frac{1}{L_c} + \frac{1}{f}$$
avec $L_{DF}=k_0 w_0^2/2$ et $f=394\ \mu\text{m}$ (distance foyer géométrique
depuis l'entrée, voir ci-dessus). $L_{c,f}$ est calculée dans le repère du
solveur (0 = foyer géométrique) puis reconvertie en repère "entrée = 0" pour
être comparée directement aux figures.

In [ ]:
def marburger_lengths(n2, w0, wavelength, n0, energy_uJ_real, delta_t, f_focus):
    """P_cr, L_c, L_c,f -- E_uJ_real est l'energie PHYSIQUE dans le verre
    (avant division par GAUSS_FLATTOP : le calcul P0=E/(1.0645*tFWHM) a besoin
    de la vraie energie, pas de la valeur reduite envoyee a run())."""
    P_cr = 3.77 * wavelength**2 / (8 * np.pi * n0 * n2)
    P0 = (energy_uJ_real * 1e-6) / (GAUSS_FLATTOP * delta_t)
    ratio = P0 / P_cr
    k0 = 2 * np.pi * n0 / wavelength
    L_DF = k0 * w0**2 / 2
    if ratio <= 0.852**2:
        return P_cr, P0, ratio, L_DF, np.nan, np.nan
    inner = (np.sqrt(ratio) - 0.852)**2 - 0.0219
    if inner <= 0:
        return P_cr, P0, ratio, L_DF, np.nan, np.nan
    L_c = 0.367 * L_DF / np.sqrt(inner)
    L_cf = 1.0 / (1.0 / L_c + 1.0 / f_focus)
    return P_cr, P0, ratio, L_DF, L_c, L_cf

for label, n2 in (("n2_default (800nm, non corrige)", N2_DEFAULT_M2W),
                  ("n2_1030nm (corrige)", N2_1030NM_M2W)):
    P_cr, P0, ratio, L_DF, L_c, L_cf = marburger_lengths(
        n2, W0_M, WAVELENGTH_M, n0_1030, ENERGY_IN_GLASS, DELTA_T_S, Z_FOCUS_GLASS_DIST_UM * 1e-6)
    z_entree_um = -Z_FOCUS_GLASS_DIST_UM + L_cf * 1e6 if np.isfinite(L_cf) else np.nan
    print(f"[{label}]")
    print(f"  P_cr = {P_cr*1e-6:.3f} MW   P_in = {P0*1e-6:.3f} MW   P_in/P_cr = {ratio:.2f}")
    print(f"  L_DF = {L_DF*1e6:.2f} um   L_c = {L_c*1e6:.2f} um   L_c,f = {L_cf*1e6:.2f} um")
    print(f"  -> collapse predit a z_sim = {L_cf*1e6 - Z_FOCUS_GLASS_DIST_UM:+.1f} um"
          f"  (z depuis l'entree = {z_entree_um:.1f} um, vs foyer geometrique a {Z_FOCUS_GLASS_DIST_UM:.0f} um)")
    print()

print("Note : L_c,f << f dans les deux cas -- la focalisation Kerr devrait gagner tres")
print("pres de l'entree, bien avant le foyer geometrique a 394 um. C'est une prediction")
print("testable directement sur Imax_z une fois le run termine (cellule 4).")

## 3. Lancer (ou recharger) la simulation

`enable_ste=True`, `tau_r=330e-15` (Mouskeftaras 2013 / Tsaturyan 2025, comme
dans `notebooksimu3juillet.ipynb`). `n2` : voir discussion ci-dessus -- ce
run utilise **`N2_1030NM_M2W`** (valeur corrigée) par défaut ; changer
`N2_CHOSEN` ci-dessous pour reproduire l'ancien run avec `N2_DEFAULT_M2W`
à la place.

**Non exécuté ici (pas d'accès GPU dans cet environnement)** -- prêt à
lancer tel quel.

In [ ]:
N2_CHOSEN = N2_1030NM_M2W   # changer en N2_DEFAULT_M2W pour reproduire l'ancien run

RUN_TAG = "n2_1030nm" if N2_CHOSEN == N2_1030NM_M2W else "n2_default"
OUT_DIR = str(OUT_ROOT / f"filament_13uJ_w3um_{RUN_TAG}")
NPZ_PATH = Path(OUT_DIR) / "result.npz"

if NPZ_PATH.exists():
    print(f"Chargement du run existant : {NPZ_PATH}")
    res = dict(np.load(NPZ_PATH, allow_pickle=True))
else:
    print(f"Lancement -> {OUT_DIR}  (Nz={NZ}, Nr={NR}, Nt={NT} -- prevoir plusieurs heures de GPU)")
    res = run(
        Nz=NZ, Nt=NT, Nr=NR,
        begin=BEGIN_M, end=END_M,
        R_factor=R_FACTOR,
        wavelength=WAVELENGTH_M,
        energy_uJ=ENERGY_SIM_UJ,
        w0=W0_M,
        delta_t=DELTA_T_S,
        n2=N2_CHOSEN,
        Ui_eV=UI_EV,
        rho_max=RHO_MAX_CM3,
        enable_ste=True,
        tau_r=330e-15,
        lambda_probe=490e-9,
        rho_t_stride=10,
        save_stride=100,
        out_dir=OUT_DIR,
        envelope="gaussian_focused",
    )
    print(f"Termine -> {OUT_DIR}/result.npz")

## Note connue : pertes d'énergie non disponibles avec ce solveur

`NewSim3juillet.py::Integrator._record()` initialise `E_plasma_z`,
`E_MPI_z`, `E_STE_z` (lignes 484-486) mais ne les remplit **jamais** --
contrairement au paquet modulaire `sim/filament_sim.py` utilisé ailleurs
dans ce rapport (Section 8, figure de pertes d'énergie style Fig. 12), ces
trois tableaux resteront à zéro dans `res`. Pas un bug introduit ici, un
état de fait du solveur uploadé : pas de figure de pertes d'énergie possible
à partir de ce run tant que ce n'est pas corrigé dans `NewSim3juillet.py`
lui-même.

## 4. Extraction des axes et helpers d'affichage

In [ ]:
z_um = np.asarray(res["z"]) * 1e6           # repere solveur : 0 = foyer geometrique
r_um = np.asarray(res["r"]) * 1e6            # deja miroir +/- (voir _results())
i_axis = int(np.argmin(np.abs(r_um)))

Imax_z  = np.asarray(res["Imax_z"])
rho_rz  = np.asarray(res["rho_rz"])          # rho_e, cm^-3
rho_s_rz = np.asarray(res["rho_s_rz"])       # rho_s (STE), cm^-3
fluence_rz = None
if "fluence_rz" in res:
    fluence_rz = np.asarray(res["fluence_rz"])

I_CLAMP = 5e13  # W/cm^2, meme convention que le reste du rapport

print(f"z   : [{z_um[0]:+.0f}, {z_um[-1]:+.0f}] um  ({len(z_um)} plans sauvegardes)")
print(f"r   : [{r_um[0]:.1f}, {r_um[-1]:.1f}] um")
print(f"I_max = {Imax_z.max():.3e} W/cm2 @ z = {z_um[np.argmax(Imax_z)]:+.1f} um")
print(f"rho_e,max on-axis = {rho_rz[:, i_axis].max():.3e} cm^-3")
print(f"rho_s,max on-axis = {rho_s_rz[:, i_axis].max():.3e} cm^-3")

## 5. Fluence (contours) -- pour observer les cycles focalisation/défocalisation

Chaque pincement du contour est un cycle candidat, comme pour la Fig. 7 du
rapport (Section 8).

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
levels = (1.0, 2.0, 5.0, 10.0, 20.0)
cs = ax.contour(z_um, r_um, fluence_rz.T, levels=levels, colors="black", linewidths=0.8)
ax.clabel(cs, inline=True, fontsize=6, fmt="%.0f J/cm2")

for label, n2 in (("L_c,f (n2 1030nm)", N2_1030NM_M2W), ("L_c,f (n2 800nm)", N2_DEFAULT_M2W)):
    _, _, _, _, _, L_cf = marburger_lengths(
        n2, W0_M, WAVELENGTH_M, n0_1030, ENERGY_IN_GLASS, DELTA_T_S, Z_FOCUS_GLASS_DIST_UM * 1e-6)
    if np.isfinite(L_cf):
        z_pred_um = L_cf * 1e6 - Z_FOCUS_GLASS_DIST_UM
        ax.axvline(z_pred_um, ls="--", lw=1, label=label)

ax.axvline(0.0, color="purple", ls=":", lw=1, label="foyer geometrique")
ax.set_xlabel("z (um, 0 = foyer geometrique)"); ax.set_ylabel("r (um)")
ax.set_title("Contours de fluence -- cycles de focalisation/defocalisation")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "fluence_contours_1030nm.png", dpi=150)

## 6. Intensité crête vs z -- avec L_c,f prédite

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(z_um, Imax_z, lw=1.6, color="black", label="I_max(z)")
ax.axhline(I_CLAMP, ls="--", color="crimson", lw=1, label=f"I_clamp~{I_CLAMP:.0e}")

for label, n2, color in (("L_c,f (n2 1030nm, corrige)", N2_1030NM_M2W, "tab:blue"),
                         ("L_c,f (n2 800nm, non corrige)", N2_DEFAULT_M2W, "tab:orange")):
    _, _, ratio, _, _, L_cf = marburger_lengths(
        n2, W0_M, WAVELENGTH_M, n0_1030, ENERGY_IN_GLASS, DELTA_T_S, Z_FOCUS_GLASS_DIST_UM * 1e-6)
    if np.isfinite(L_cf):
        z_pred_um = L_cf * 1e6 - Z_FOCUS_GLASS_DIST_UM
        ax.axvline(z_pred_um, ls="--", lw=1.2, color=color,
                  label=f"{label}  (P/Pcr={ratio:.1f})")

ax.set_yscale("log")
ax.set_xlabel("z (um, 0 = foyer geometrique)"); ax.set_ylabel("Peak intensity (W/cm2)")
ax.set_title("Intensite crete vs z -- L_c,f predite vs collapse observe")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "peak_intensity_vs_Lc_1030nm.png", dpi=150)

## 7. Densité électronique libre (rho_e) et excitons auto-piégés (rho_s) on-axis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(z_um, np.clip(rho_rz[:, i_axis], 1e-3, None), lw=1.6, color="black", label="rho_e (libres)")
ax.plot(z_um, np.clip(rho_s_rz[:, i_axis], 1e-3, None), lw=1.6, color="tab:blue", label="rho_s (STE)")
ax.axhline(RHO_MAX_CM3, ls=":", color="gray", lw=1, label=f"rho_max={RHO_MAX_CM3:.1e}")
ax.set_yscale("log")
ax.set_xlabel("z (um, 0 = foyer geometrique)"); ax.set_ylabel("On-axis rho (cm^-3)")
ax.set_title("Densite electronique libre vs excitons auto-pieges, on-axis")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "rho_e_rho_s_vs_z_1030nm.png", dpi=150)

## 8. Compter les cycles de refocalisation

Même critère que pour la run longue boîte de la Section 8 : chaque maximum
local de `Imax_z` au-dessus de `I_clamp`, au-delà du premier, est un cycle
supplémentaire.

In [ ]:
peaks_idx, _ = find_peaks(Imax_z, height=I_CLAMP)
print(f"{len(peaks_idx)} maximum(aux) local(aux) au-dessus de I_clamp:")
for i, ip in enumerate(peaks_idx):
    print(f"  cycle {i+1}: z = {z_um[ip]:+8.1f} um   I_peak = {Imax_z[ip]:.3e} W/cm2")
if len(peaks_idx) >= 2:
    spacings = np.diff(z_um[peaks_idx])
    print("Espacements successifs (um):", np.round(spacings, 1))

if len(peaks_idx) >= 1:
    z_first_collapse = z_um[peaks_idx[0]]
    for label, n2 in (("n2_1030nm", N2_1030NM_M2W), ("n2_default", N2_DEFAULT_M2W)):
        _, _, _, _, _, L_cf = marburger_lengths(
            n2, W0_M, WAVELENGTH_M, n0_1030, ENERGY_IN_GLASS, DELTA_T_S, Z_FOCUS_GLASS_DIST_UM * 1e-6)
        z_pred = L_cf * 1e6 - Z_FOCUS_GLASS_DIST_UM if np.isfinite(L_cf) else np.nan
        print(f"[{label}] premier collapse observe a z={z_first_collapse:+.1f} um "
              f"vs predit a z={z_pred:+.1f} um  (ecart {z_first_collapse - z_pred:+.1f} um)")

## 9. Figures pour `main.tex`

Toutes les figures ci-dessus sont déjà sauvegardées dans `FIG_DIR`
(`runs_1030nm/filament_13uJ_w3um_<tag>/../figures_for_report/`). Résumé
final pour copier-coller dans le rapport (Section 8) une fois le run
terminé.

In [ ]:
print("Figures sauvegardees dans:", FIG_DIR.resolve())
for f in sorted(FIG_DIR.glob("*.png")):
    print(" -", f.name)

print()
print(f"Run: 1030 nm, {ENERGY_INPUT_UJ:.1f} uJ incident ({ENERGY_IN_GLASS:.2f} uJ dans le verre), "
      f"w0={W0_M*1e6:.1f} um, FWHM={DELTA_T_S*1e15:.0f} fs, n2={N2_CHOSEN:.2e} m2/W ({RUN_TAG})")
print(f"I_max = {Imax_z.max():.3e} W/cm2 @ z={z_um[np.argmax(Imax_z)]:+.1f} um")
print(f"{len(peaks_idx)} cycle(s) de refocalisation detecte(s) au-dessus de I_clamp={I_CLAMP:.0e} W/cm2")